## Setup

In [182]:
import sqlite3
import pandas as pd

def q(sql):
    con = sqlite3.connect("sts2.db")
    try:
        return pd.read_sql(sql, con)
    finally:
        con.close()

## Which character wins most?

In [183]:
q("""
SELECT character, COUNT(*) AS n_runs, AVG(win) AS winrate
FROM runs
GROUP BY character
""")

,character,n_runs,winrate
0,CHARACTER.DEFECT,63,0.111111
1,CHARACTER.IRONCLAD,69,0.188406
2,CHARACTER.NECROBINDER,74,0.175676
3,CHARACTER.REGENT,76,0.184211
4,CHARACTER.SILENT,58,0.120690


## How does win rate change by ascension?

In [184]:
q("""
  SELECT REPLACE(character, 'CHARACTER.', '') AS character,
       ascension,
       COUNT(*) AS n_runs,
       ROUND(AVG(win), 3) AS winrate
FROM runs
GROUP BY character, ascension
ORDER BY character, ascension
""")

,character,ascension,n_runs,winrate
0,DEFECT,9,2,0.500
1,DEFECT,10,61,0.098
2,IRONCLAD,10,69,0.188
3,NECROBINDER,7,6,0.167
4,NECROBINDER,8,2,0.500
5,NECROBINDER,9,4,0.250
6,NECROBINDER,10,62,0.161
7,REGENT,8,1,1.000
8,REGENT,9,12,0.083
9,REGENT,10,63,0.190


## Where do I die?

In [185]:
q("""
SELECT REPLACE(encounter,'ENCOUNTER.', '') as ENCOUNTER, act, count(*) as deaths
FROM floors
JOIN runs on runs.run_id = floors.run_id
WHERE floor = final_floor AND win = 0
GROUP BY act, encounter
ORDER BY deaths DESC
""")

,ENCOUNTER,act,deaths
0,QUEEN_BOSS,3,30
1,KAISER_CRAB_BOSS,2,23
2,AEONGLASS_BOSS,3,21
3,WATERFALL_GIANT_BOSS,1,16
4,DECIMILLIPEDE_ELITE,2,16
5,THE_INSATIABLE_BOSS,2,16
6,TEST_SUBJECT_BOSS,3,16
7,INFESTED_PRISMS_ELITE,2,15
8,KNOWLEDGE_DEMON_BOSS,2,11
9,CEREMONIAL_BEAST_BOSS,1,10


## Which cards do I pick from combat rewards?

In [186]:
q("""
SELECT REPLACE(co.card_id, 'CARD.', '') AS card,
       COUNT(*)                AS offered,
       SUM(co.was_picked)      AS picked,
       ROUND(AVG(co.was_picked), 3) AS pickrate
FROM card_offers co
JOIN runs r ON r.run_id = co.run_id
WHERE co.source = 'combat'
  AND r.schema_version = 10
GROUP BY co.card_id
HAVING COUNT(*) >= 10
ORDER BY pickrate DESC
""")

,card,offered,picked,pickrate
0,FORGOTTEN_RITUAL,11,10,0.909
1,FOOTWORK,11,10,0.909
2,FLAME_BARRIER,17,15,0.882
3,PARTICLE_WALL,13,10,0.769
4,LETHALITY,13,10,0.769
...,...,...,...,...
262,LUNAR_BLAST,13,0,0.000
263,JUGGLING,17,0,0.000
264,CRESCENT_SPEAR,51,0,0.000
265,CHAOS,11,0,0.000


## Which relics do I take from Ancients?

In [187]:
q("""
  SELECT REPLACE(f.encounter, 'EVENT.', '') AS ancient,
       REPLACE(ro.relic_id, 'RELIC.', '') AS relic,
       COUNT(*) AS offered,
       ROUND(AVG(ro.was_picked), 3) AS pickrate
FROM relic_offers ro
JOIN floors f ON f.run_id = ro.run_id AND f.floor = ro.floor
WHERE ro.source = 'ancient'
GROUP BY f.encounter, ro.relic_id
HAVING COUNT(*) >= 5
ORDER BY ancient, pickrate DESC
  """)

,ancient,relic,offered,pickrate
0,DARV,DUSTY_TOME,27,0.704
1,DARV,PANDORAS_BOX,11,0.545
2,DARV,ASTROLABE,14,0.357
3,DARV,BLACK_STAR,16,0.313
4,DARV,EMPTY_CAGE,11,0.273
...,...,...,...,...
93,VAKUU,PRESERVED_FOG,18,0.333
94,VAKUU,LORDS_PARASOL,9,0.222
95,VAKUU,BLOOD_SOAKED_ROSE,15,0.200
96,VAKUU,WHISPERING_EARRING,15,0.133


## What do I do at rest sites?

In [188]:
display(q("SELECT COUNT(*) FROM rest_choices"))
display(q("SELECT choice, COUNT(*) AS n FROM rest_choices GROUP BY choice ORDER BY n DESC"))

,COUNT(*)
0,2099


,choice,n
0,SMITH,1471
1,HEAL,599
2,COOK,15
3,KINDLE,7
4,LIFT,4
5,CLONE,2
6,HATCH,1


## Where do relic offers come from?

In [189]:
q("""
SELECT source, COUNT(*) AS offered, SUM(was_picked) AS picked
FROM relic_offers
GROUP BY source
ORDER BY offered DESC
""")

,source,offered,picked
0,shop,2240,394
1,ancient,2205,735
2,elite,1184,1184
3,treasure,743,743
4,event,493,447
5,monster,4,4
6,boss,2,2
7,rest_site,1,1


In [190]:
q("""
SELECT ro.source, ro.relic_id, f.encounter, f.event_id, f.map_point_type
FROM relic_offers ro
JOIN floors f ON f.run_id = ro.run_id AND f.floor = ro.floor
WHERE ro.source IN ('monster', 'boss', 'rest_site')
""")

,source,relic_id,encounter,event_id,map_point_type
0,rest_site,RELIC.BYRDPIP,NaN,None,rest_site
1,monster,RELIC.BAG_OF_PREPARATION,ENCOUNTER.THE_OBSCURA_NORMAL,None,monster
2,monster,RELIC.LANTERN,ENCOUNTER.DEVOTED_SCULPTOR_WEAK,None,monster
3,monster,RELIC.ANCHOR,ENCOUNTER.CHOMPERS_NORMAL,None,monster
4,monster,RELIC.JOSS_PAPER,ENCOUNTER.SLUMBERING_BEETLE_NORMAL,None,monster
5,boss,RELIC.PEN_NIB,ENCOUNTER.LAGAVULIN_MATRIARCH_BOSS,None,boss
6,boss,RELIC.LETTER_OPENER,ENCOUNTER.LAGAVULIN_MATRIARCH_BOSS,None,boss


In [191]:
q("""
SELECT COUNT(*) AS times_offered,
       SUM(was_picked) AS times_picked
FROM relic_offers
WHERE relic_id = 'RELIC.PAELS_WING'
""")

,times_offered,times_picked
0,16,4


In [192]:
display(q("SELECT COUNT(*) FROM final_deck"))      # expect 9,140
display(q("SELECT COUNT(*) FROM final_relics"))    # expect 3,906
display(q("""SELECT COUNT(DISTINCT run_id) FROM final_relics
             WHERE relic_id = 'RELIC.PAELS_WING'""")) # expect 4, matching relic_offers

,COUNT(*)
0,9140


,COUNT(*)
0,3906


,COUNT(DISTINCT run_id)
0,4


## Which cards show up in winning decks?

In [193]:
q("""
SELECT REPLACE(fd.card_id, 'CARD.', '') AS card,
       COUNT(DISTINCT fd.run_id) AS runs_with_card,
       COUNT(DISTINCT CASE WHEN r.win = 1 THEN r.run_id END) AS wins,
       ROUND(1.0 * COUNT(DISTINCT CASE WHEN r.win = 1 THEN r.run_id END)
                 / COUNT(DISTINCT fd.run_id), 3) AS winrate
FROM runs r
JOIN final_deck fd ON r.run_id = fd.run_id
WHERE r.schema_version = 10
GROUP BY fd.card_id
HAVING COUNT(DISTINCT fd.run_id) >= 15 
ORDER BY winrate DESC
""")

,card,runs_with_card,wins,winrate
0,PIERCING_WAIL,15,4,0.267
1,BLOODLETTING,15,4,0.267
2,GATHER_LIGHT,20,5,0.250
3,COSMIC_INDIFFERENCE,16,4,0.250
4,CLUMSY,23,5,0.217
5,FALLING_STAR,25,5,0.200
6,DEFY,20,4,0.200
7,DECAY,15,3,0.200
8,POMMEL_STRIKE,16,3,0.188
9,NEUTRALIZE,22,4,0.182


## Which cards beat their character's baseline win rate?

In [194]:
q("""
WITH char_base AS (            -- one row per character: runs and winrate
    SELECT character, COUNT(*) AS char_runs, AVG(win) AS char_winrate
    FROM runs
    GROUP BY character
),
card_runs AS (                 -- one row per character + card
    SELECT r.character, fd.card_id,
           COUNT(DISTINCT r.run_id) AS runs_with_card,
           COUNT(DISTINCT CASE WHEN r.win = 1 THEN r.run_id END) AS wins
    FROM runs r
    JOIN final_deck fd ON fd.run_id = r.run_id
    GROUP BY r.character, fd.card_id
)
SELECT REPLACE(c.character, 'CHARACTER.', '') AS character,
       REPLACE(c.card_id, 'CARD.', '') AS card,
       c.runs_with_card,
       ROUND(1.0 * c.wins / c.runs_with_card, 3) AS winrate,
       ROUND(b.char_winrate, 3) AS char_winrate,
       ROUND(1.0 * c.wins / c.runs_with_card - b.char_winrate, 3) AS lift
FROM card_runs c
JOIN char_base b ON b.character = c.character
WHERE 1.0 * c.runs_with_card / b.char_runs < 0.9      -- drop starter cards
  AND c.card_id <> 'CARD.ASCENDERS_BANE'
  AND c.runs_with_card >= 15
ORDER BY lift DESC
""")

,character,card,runs_with_card,winrate,char_winrate,lift
0,IRONCLAD,FEEL_NO_PAIN,20,0.500,0.188,0.312
1,NECROBINDER,INVOKE,17,0.471,0.176,0.295
2,NECROBINDER,NEUROSURGE,16,0.438,0.176,0.262
3,REGENT,SPECTRUM_SHIFT,23,0.435,0.184,0.251
4,NECROBINDER,ENFEEBLING_TOUCH,22,0.409,0.176,0.233
...,...,...,...,...,...,...
126,REGENT,PARTICLE_WALL,23,0.087,0.184,-0.097
127,DEFECT,LEAP,15,0.000,0.111,-0.111
128,NECROBINDER,SCULPTING_STRIKE,16,0.063,0.176,-0.113
129,REGENT,COLLISION_COURSE,15,0.067,0.184,-0.118


## When do I heal?

In [195]:
q("""
WITH floors_prev AS (
    SELECT run_id, floor, hp, max_hp,
           LAG(hp)     OVER (PARTITION BY run_id ORDER BY floor) AS hp_before,
           LAG(max_hp) OVER (PARTITION BY run_id ORDER BY floor) AS max_hp_before
    FROM floors
)
SELECT rc.run_id, rc.floor, rc.choice,
       fp.hp_before, fp.max_hp_before, fp.hp AS hp_after,
       ROUND(1.0 * fp.hp_before / fp.max_hp_before, 2) AS hp_pct
FROM rest_choices rc
JOIN floors_prev fp ON fp.run_id = rc.run_id AND fp.floor = rc.floor
LIMIT 20


""")


,run_id,floor,choice,hp_before,max_hp_before,hp_after,hp_pct
0,1774766158,7,SMITH,56,75,56,0.75
1,1774766158,9,SMITH,49,75,49,0.65
2,1774766158,12,SMITH,40,75,40,0.53
3,1774766158,16,HEAL,27,75,64,0.36
4,1774766158,24,SMITH,66,75,66,0.88
5,1774766158,29,SMITH,51,75,51,0.68
6,1774766158,32,SMITH,33,75,33,0.44
7,1774766158,40,COOK,54,75,63,0.72
8,1774766158,43,COOK,38,84,47,0.45
9,1774766158,47,HEAL,23,93,65,0.25


In [196]:
q("""
  WITH floors_prev AS (
    SELECT run_id, floor, hp, max_hp,
           LAG(hp)     OVER (PARTITION BY run_id ORDER BY floor) AS hp_before,
           LAG(max_hp) OVER (PARTITION BY run_id ORDER BY floor) AS max_hp_before
    FROM floors
)
SELECT ROUND(1.0 * fp.hp_before / fp.max_hp_before, 1) AS hp_bucket,
       COUNT(*) AS visits,
       ROUND(AVG(CASE WHEN rc.choice = 'HEAL' THEN 1.0 ELSE 0 END), 2) AS heal_share
FROM rest_choices rc
JOIN floors_prev fp ON fp.run_id = rc.run_id AND fp.floor = rc.floor
AND fp.max_hp_before > 5
GROUP BY hp_bucket
ORDER BY hp_bucket
""")

,hp_bucket,visits,heal_share
0,0.0,32,0.97
1,0.1,103,0.86
2,0.2,172,0.81
3,0.3,244,0.54
4,0.4,279,0.33
5,0.5,339,0.16
6,0.6,273,0.11
7,0.7,288,0.05
8,0.8,162,0.04
9,0.9,108,0.01


In [197]:
q("""
WITH floors_prev AS (
    SELECT run_id, floor, hp, max_hp,
           LAG(hp)     OVER (PARTITION BY run_id ORDER BY floor) AS hp_before,
           LAG(max_hp) OVER (PARTITION BY run_id ORDER BY floor) AS max_hp_before
    FROM floors
),
visit_size AS (                          
    SELECT run_id, floor,
           COUNT(*) AS n_choices
    FROM rest_choices
    GROUP BY run_id, floor 
)
SELECT ROUND(1.0 * fp.hp_before / fp.max_hp_before, 1) AS hp_bucket,
       COUNT(*) AS visits,
       ROUND(AVG(CASE WHEN rc.choice = 'HEAL' THEN 1.0 ELSE 0 END), 2) AS heal_share
FROM rest_choices rc
JOIN floors_prev fp ON fp.run_id = rc.run_id AND fp.floor = rc.floor
JOIN visit_size vs ON vs.run_id = fp.run_id AND vs.floor = fp.floor
WHERE fp.max_hp_before > 5
  AND n_choices = 1
GROUP BY hp_bucket
ORDER BY hp_bucket
""")

,hp_bucket,visits,heal_share
0,0.0,32,0.97
1,0.1,101,0.87
2,0.2,162,0.83
3,0.3,236,0.54
4,0.4,273,0.33
5,0.5,331,0.15
6,0.6,257,0.08
7,0.7,274,0.03
8,0.8,152,0.01
9,0.9,106,0.00


## What do I buy at shops?

In [198]:
q("""SELECT COUNT(*), SUM(gold_spent), SUM(cards_bought), SUM(relics_bought),
            SUM(potions_bought), SUM(cards_removed) FROM shop_visits""")

,COUNT(*),SUM(gold_spent),SUM(cards_bought),SUM(relics_bought),SUM(potions_bought),SUM(cards_removed)
0,741,174141,729,394,293,339


In [199]:
q("""
SELECT COUNT(*)                          AS visits,
       ROUND(AVG(cards_removed  > 0), 2) AS removal,
       ROUND(AVG(cards_bought   > 0), 2) AS card,
       ROUND(AVG(relics_bought  > 0), 2) AS relic,
       ROUND(AVG(potions_bought > 0), 2) AS potion,
       ROUND(AVG(gold_spent = 0), 2)     AS spent_nothing
FROM shop_visits
""")

,visits,removal,card,relic,potion,spent_nothing
0,741,0.46,0.66,0.42,0.33,0.04


In [200]:
q("""
SELECT COUNT(*)                          AS visits,
       ROUND(AVG(cards_removed  > 0), 2) AS removal,
       ROUND(AVG(cards_bought   > 0), 2) AS card,
       ROUND(AVG(relics_bought  > 0), 2) AS relic,
       ROUND(AVG(potions_bought > 0), 2) AS potion,
       ROUND(AVG(gold_spent = 0), 2)     AS spent_nothing
FROM shop_visits sv

""")

,visits,removal,card,relic,potion,spent_nothing
0,741,0.46,0.66,0.42,0.33,0.04


## How does shop spending change by act?

In [201]:
q("""
SELECT f.act,
       COUNT(*)                                          AS visits,
       ROUND(AVG(s.gold_before))                         AS avg_gold_on_arrival,
       ROUND(AVG(s.gold_spent))                          AS avg_spent,
       ROUND(1.0 * SUM(s.gold_spent) / SUM(s.gold_before), 2) AS share_gold_spent,
       ROUND(AVG(s.cards_removed  > 0), 2)               AS removal,
       ROUND(AVG(s.cards_bought   > 0), 2)               AS card,
       ROUND(AVG(s.relics_bought  > 0), 2)               AS relic,
       ROUND(AVG(s.potions_bought > 0), 2)               AS potion
FROM shop_visits s
JOIN floors f ON f.run_id = s.run_id AND f.floor = s.floor
GROUP BY f.act
ORDER BY f.act
""")


,act,visits,avg_gold_on_arrival,avg_spent,share_gold_spent,removal,card,relic,potion
0,1,319,225.0,197.0,0.88,0.36,0.79,0.29,0.28
1,2,253,300.0,267.0,0.89,0.54,0.59,0.51,0.37
2,3,169,321.0,259.0,0.81,0.51,0.50,0.54,0.38


## Which shop relics do I buy?

In [202]:
q("""
SELECT REPLACE(relic_id, 'RELIC.', '') AS relic,
       COUNT(*)                        AS offered,
       SUM(was_picked)                 AS bought,
       ROUND(AVG(was_picked), 3)       AS buyrate
FROM relic_offers
WHERE source = 'shop'
GROUP BY relic_id
HAVING COUNT(*) >= 10
ORDER BY buyrate DESC
""")

,relic,offered,bought,buyrate
0,ANCHOR,22,13,0.591
1,BAG_OF_PREPARATION,26,14,0.538
2,MINIATURE_TENT,20,10,0.500
3,MEMBERSHIP_CARD,26,13,0.500
4,CANDELABRA,26,11,0.423
...,...,...,...,...
77,JUZU_BRACELET,23,0,0.000
78,JOSS_PAPER,15,0,0.000
79,GIRYA,10,0,0.000
80,DINGY_RUG,20,0,0.000


## Did the patch change my results?

In [203]:
q("""
  SELECT run_id, build_id, win, final_floor,
       AVG(win) OVER (ORDER BY run_id
                      ROWS BETWEEN 29 PRECEDING AND CURRENT ROW) AS rolling_winrate,
       AVG(final_floor) OVER (ORDER BY run_id
                      ROWS BETWEEN 29 PRECEDING AND CURRENT ROW) AS rolling_floor
FROM runs
WHERE ascension = 10
ORDER BY run_id
  """)

,run_id,build_id,win,final_floor,rolling_winrate,rolling_floor
0,1774948056,v0.101.0,0,28,0.000000,28.000000
1,1775044357,v0.101.0,0,9,0.000000,18.500000
2,1775044627,v0.101.0,0,25,0.000000,20.666667
3,1775045631,v0.101.0,0,49,0.000000,27.750000
4,1775105112,v0.101.0,0,24,0.000000,27.000000
...,...,...,...,...,...,...
297,1791079648,v0.111.0,0,24,0.133333,33.000000
298,1791080776,v0.111.0,0,49,0.133333,33.033333
299,1791205815,v0.111.0,0,33,0.133333,33.633333
300,1791353788,v0.111.0,0,49,0.133333,34.833333


In [204]:
q("""
SELECT build_id, COUNT(*) AS runs,
       ROUND(AVG(win), 2) AS winrate,
       ROUND(AVG(final_floor), 1) AS avg_floor
FROM runs
WHERE ascension = 10
GROUP BY build_id
ORDER BY MIN(run_id)
""")

,build_id,runs,winrate,avg_floor
0,v0.101.0,7,0.14,33.3
1,v0.102.0,8,0.00,23.1
2,v0.103.0,17,0.12,36.8
3,v0.103.1,1,0.00,48.0
4,v0.103.2,10,0.20,33.2
5,v0.104.0,55,0.16,32.6
6,v0.105.0,4,0.25,27.0
7,v0.105.1,34,0.21,30.8
8,v0.106.0,5,0.20,35.2
9,v0.106.1,9,0.22,40.0


In [205]:
q("""
WITH v111 AS (
    SELECT win, final_floor,
           NTILE(2) OVER (ORDER BY run_id) AS half
    FROM runs
    WHERE ascension = 10 AND build_id = 'v0.111.0'
)
SELECT half,
       COUNT(*) AS runs,
       ROUND(AVG(win), 2) AS winrate,
       ROUND(AVG(CASE WHEN win = 0 THEN final_floor END), 1) AS avg_floor_on_loss
FROM v111
GROUP BY half
""")

,half,runs,winrate,avg_floor_on_loss
0,1,71,0.14,31.1
1,2,70,0.14,33.8


In [206]:
from scipy import stats

df = q("""
SELECT final_floor,
       NTILE(2) OVER (ORDER BY run_id) AS half
FROM runs
WHERE ascension = 10 AND build_id = 'v0.111.0'
""")
# keep losses only, after the halves are assigned
losses = df.merge(q("SELECT final_floor FROM runs LIMIT 0"), how="left")  # placeholder, see note

In [207]:
from scipy import stats

df = q("""
SELECT win, final_floor,
       NTILE(2) OVER (ORDER BY run_id) AS half
FROM runs
WHERE ascension = 10 AND build_id = 'v0.111.0'
""")
losses = df[df["win"] == 0]

display(losses.groupby("half")["final_floor"].agg(["count", "mean", "std"]))

h1 = losses.loc[losses["half"] == 1, "final_floor"]
h2 = losses.loc[losses["half"] == 2, "final_floor"]

,count,mean,std
half,,,
1,61,31.081967,14.201285
2,60,33.816667,14.657088


## Do "?" rooms follow the expected odds?

In [208]:
floors = q("""SELECT f.run_id, f.floor, f.act, f.map_point_type, f.room_type, f.event_id
              FROM floors f ORDER BY f.run_id, f.floor""")
juzu = q("""SELECT run_id, MIN(floor_added) AS f FROM final_relics
            WHERE relic_id = 'RELIC.JUZU_BRACELET' GROUP BY run_id""").set_index("run_id")["f"].to_dict()

BASE = {"monster": 0.10, "treasure": 0.02, "shop": 0.03}
rows, prev = [], None
for r in floors.itertuples():
    if prev is None or r.run_id != prev.run_id or r.act != prev.act:
        k = {t: 0 for t in BASE}               # new run or new act: all counters reset
    if r.map_point_type == "unknown":
        outcome = "event" if pd.notna(r.event_id) else r.room_type
        if outcome not in BASE:
            outcome = "event"
        if not (r.run_id in juzu and juzu[r.run_id] < r.floor):
            p = {t: BASE[t] * (k[t] + 1) for t in BASE}
            # TODO: shop chance is 0 if the previous floor was a shop
            if prev is not None and prev.run_id == r.run_id and prev.act == r.act:
                if prev.room_type == "shop":
                    p["shop"] = 0
            p["event"] = 1 - sum(p.values())
            rows.append({**{f"p_{t}": v for t, v in p.items()}, "outcome": outcome,
                         "k_monster": k["monster"]})
        # TODO: update counters: reset to 0 for the outcome that happened, +1 for the others
        for t in BASE:
            if t == outcome:
                k[t] = 0
            else:
                k[t] += 1
        
    prev = r

sim = pd.DataFrame(rows)

In [209]:
display(len(sim))   # expect 2,053

# observed vs expected counts per outcome
order = ["event", "monster", "shop", "treasure"]
observed = sim["outcome"].value_counts().reindex(order, fill_value=0)
expected = sim[[f"p_{t}" for t in order]].sum().values
display(pd.DataFrame({"observed": observed.values,
                      "expected": expected.round(1)}, index=order))

# chi-square goodness-of-fit
from scipy import stats
print(stats.chisquare(observed.values, expected))   # expect statistic ≈ 2.17

# monster staircase
stair = sim.groupby(sim["k_monster"].clip(upper=5)).agg(
    rooms=("outcome", "size"),
    expected=("p_monster", "mean"),
    observed=("outcome", lambda s: (s == "monster").mean()),
)
display(stair.round(3))

2053

,observed,expected
event,1486,1459.4
monster,372,387.2
shop,117,118.7
treasure,78,87.7


Power_divergenceResult(statistic=np.float64(2.172521040476024), pvalue=np.float64(0.5373811302450155))


,rooms,expected,observed
k_monster,,,
0,906,0.1,0.098
1,650,0.2,0.180
2,351,0.3,0.296
3,122,0.4,0.418
4,19,0.5,0.421
5,5,0.6,0.600


## Export data for the dashboard

In [210]:
from pathlib import Path
out = Path("exports")
out.mkdir(exist_ok=True)

exports = {
    "deaths": """
        SELECT f.act,
               f.room_type,
               REPLACE(f.encounter, 'ENCOUNTER.', '') AS encounter,
               COUNT(*) AS deaths
        FROM floors f
        JOIN runs r ON r.run_id = f.run_id
        WHERE f.floor = r.final_floor AND r.win = 0
        GROUP BY f.act, f.room_type, f.encounter
        ORDER BY deaths DESC
        """,
    "heal_threshold": """
        WITH floors_prev AS (
            SELECT run_id, floor,
                   LAG(hp)     OVER (PARTITION BY run_id ORDER BY floor) AS hp_before,
                   LAG(max_hp) OVER (PARTITION BY run_id ORDER BY floor) AS max_hp_before,
                   LEAD(room_type) OVER (PARTITION BY run_id ORDER BY floor) AS next_room
            FROM floors
        ),
        visit_size AS (
            SELECT run_id, floor, COUNT(*) AS n_choices
            FROM rest_choices
            GROUP BY run_id, floor
        )
        SELECT CAST(10 * fp.hp_before / fp.max_hp_before AS INT) * 10 AS hp_pct_floor,  -- 0 = 0-9%, 10 = 10-19%, ...
               CASE WHEN fp.next_room = 'boss' THEN 'Before boss' ELSE 'Other' END AS before_boss,
               COUNT(*) AS visits,
               ROUND(AVG(CASE WHEN rc.choice = 'HEAL' THEN 1.0 ELSE 0 END), 2) AS heal_share
        FROM rest_choices rc
        JOIN floors_prev fp ON fp.run_id = rc.run_id AND fp.floor = rc.floor
        JOIN visit_size vs  ON vs.run_id = rc.run_id AND vs.floor = rc.floor
        WHERE fp.max_hp_before > 5
          AND vs.n_choices = 1
        GROUP BY hp_pct_floor, before_boss
        ORDER BY before_boss, hp_pct_floor
        """,
    "shop_by_act": """
        SELECT f.act,
               COUNT(*)                                          AS visits,
               ROUND(AVG(s.gold_before))                         AS avg_gold_on_arrival,
               ROUND(AVG(s.gold_spent))                          AS avg_spent,
               ROUND(1.0 * SUM(s.gold_spent) / SUM(s.gold_before), 2) AS share_gold_spent,
               ROUND(AVG(s.cards_removed  > 0), 2)               AS removal,
               ROUND(AVG(s.cards_bought   > 0), 2)               AS card,
               ROUND(AVG(s.relics_bought  > 0), 2)               AS relic,
               ROUND(AVG(s.potions_bought > 0), 2)               AS potion
        FROM shop_visits s
        JOIN floors f ON f.run_id = s.run_id AND f.floor = s.floor
        GROUP BY f.act
        ORDER BY f.act
        """,
    "rolling_a10": """
        SELECT ROW_NUMBER() OVER (ORDER BY run_id) AS run_no,
               ROW_NUMBER() OVER (ORDER BY run_id) >= 30 AS full_window,  -- rolling_* are true 30-run averages only when 1
               datetime(run_id, 'unixepoch')       AS played_at,  -- UTC; Bangkok = +7h
               run_id, build_id, win, final_floor,
               AVG(win) OVER (ORDER BY run_id
                              ROWS BETWEEN 29 PRECEDING AND CURRENT ROW) AS rolling_winrate,
               AVG(final_floor) OVER (ORDER BY run_id
                              ROWS BETWEEN 29 PRECEDING AND CURRENT ROW) AS rolling_floor
        FROM runs
        WHERE ascension = 10
        ORDER BY run_id
        """,
}
for name, sql in exports.items():
    q(sql).to_csv(out / f"{name}.csv", index=False)

stair.round(3).reset_index().to_csv(out / "unknown_staircase.csv", index=False)  # needs the simulation cell run first
q("""
SELECT r.run_id,
       REPLACE(r.character, 'CHARACTER.', '') AS character,
       r.ascension, r.win,
       r.run_time_min, r.final_floor, r.build_id,
       CASE WHEN r.win = 0 THEN f.act END       AS death_act,
       CASE WHEN r.win = 0 THEN f.room_type END AS death_room,
       CASE WHEN r.win = 0 THEN REPLACE(f.encounter, 'ENCOUNTER.', '') END AS killed_by
FROM runs r
JOIN floors f ON f.run_id = r.run_id AND f.floor = r.final_floor
""").to_csv(out / "runs.csv", index=False)
print(sorted(p.name for p in out.glob("*.csv")))

['deaths.csv', 'heal_threshold.csv', 'rolling_a10.csv', 'runs.csv', 'shop_by_act.csv', 'unknown_staircase.csv']
